In [ ]:
import pandas as pd 
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt #brauchen wir das überhaupt?
import scipy.interpolate as interp #und das?
import os
import zipfile
import scipy.signal as signal



In [37]:
def verarbeite_und_resample_datei(datei_pfad):
    # 1. Datei einlesen
    df = pd.read_csv(datei_pfad)
    
    # --- SCHRITT 1: DATENBEREINIGUNG ---
    df = df[df['seconds_elapsed'] >= 0].copy()
    if len(df) < 20: # Etwas mehr Zeilen fordern, damit der Filter genug Daten hat
        return None
    
    # --- SCHRITT 2: QUALITÄTSPRÜFUNG ---
    zeit_lücken = df['seconds_elapsed'].diff()
    if zeit_lücken.median() > 0.025:
        return None
    
    # --- SCHRITT 2: RESAMPLING AUF 50 HZ ---
    df.index = pd.to_datetime(df['time'], unit='ns')
    df_resampled = df.resample('20ms').mean().interpolate(method='linear')
    
    # --- SCHRITT 3: SIGNALFILTERUNG (ROBUSTER BUTTERWORTH) ---
    try:
        # Wir zwingen die Grenzfrequenz auf einen stabilen Wert (z.B. 15 Hz),
        # falls 20 Hz zu nah an der Grenze liegt
        b, a = signal.butter(4, 15.0, btype='low', analog=False, fs=50.0)
        
        df_resampled['x'] = signal.filtfilt(b, a, df_resampled['x'])
        df_resampled['y'] = signal.filtfilt(b, a, df_resampled['y'])
        df_resampled['z'] = signal.filtfilt(b, a, df_resampled['z'])
    except Exception as e:
        print(f"⚠️ Filter-Fehler bei Ordner {datei_pfad.parent.name}: {e}")
        return None
    
    return df_resampled

print("Funktion erfolgreich aktualisiert!")

Funktion erfolgreich aktualisiert!


In [38]:
# Hier steckt dein alter Code mit drin, sucht aber gezielt nach Accelerometer
daten_ordner = Path("../data")
dateien = list(daten_ordner.glob("**/*.csv"))
accelerometer_dateien = [d for d in dateien if d.name == "Accelerometer.csv"]

if accelerometer_dateien:
    test_datei = accelerometer_dateien[0]
    print(f"Teste Datei: {test_datei.parent.name}/{test_datei.name}")
    
    df_bereinigt = verarbeite_und_resample_datei(test_datei)
    
    if df_bereinigt is not None:
        print("\n--- ERFOLG! ---")
        print("So sieht die Tabelle nach dem 50 Hz Resampling aus:")
        print(df_bereinigt[['x', 'y', 'z', 'seconds_elapsed']].head())
else:
    print("Keine 'Accelerometer.csv' Dateien gefunden. Bitte Pfad prüfen!")

Teste Datei: Brusttasche_Hinten_-2026-05-23_10-59-39 2/Accelerometer.csv

--- ERFOLG! ---
So sieht die Tabelle nach dem 50 Hz Resampling aus:
                                x         y         z  seconds_elapsed
time                                                                  
2026-05-23 10:59:39.640 -0.574367  0.629865 -0.369606         0.067108
2026-05-23 10:59:39.660 -0.517170  0.746838  1.134372         0.082026
2026-05-23 10:59:39.680 -0.440406  0.941964  2.125958         0.101918
2026-05-23 10:59:39.700 -0.423131  0.974480  2.101734         0.121809
2026-05-23 10:59:39.720 -0.568451  0.722129  1.235911         0.141700


In [39]:
# Parameter für das Sliding Window definieren
FENSTER_LAENGE = 100  # 2 Sekunden * 50 Hz = 100 Datenpunkte
SCHRITT_WEITE = 50    # 50% Überlappung = Sprung um 50 Punkte

alle_fenster = []
alle_labels = []

print("Starte das Windowing über alle Dateien...")

# Wir gehen durch jede einzelne Accelerometer-Datei
for d in accelerometer_dateien:
    # Unsere fertige Funktion aus Block 2 aufrufen
    df_bereinigt = verarbeite_und_resample_datei(d)
    
    # Falls die Datei die Qualitätsprüfung bestanden hat
    if df_bereinigt is not None:
        # Wir brauchen die Achsen als reines Zahlen-Array
        sensor_daten = df_bereinigt[['x', 'y', 'z']].values
        
        # Name des Ordners ist unser Label (z.B. Sturztyp oder ADL)
        label_name = d.parent.name
        
        # Nun schneiden wir die Daten in sich überlappende 2-Sekunden-Fenster
        anzahl_punkte = len(sensor_daten)
        for start_idx in range(0, anzahl_punkte - FENSTER_LAENGE + 1, SCHRITT_WEITE):
            end_idx = start_idx + FENSTER_LAENGE
            
            # Ein Fenster ausschneiden (Form: 100 Zeilen, 3 Spalten)
            fenster = sensor_daten[start_idx:end_idx]
            
            alle_fenster.append(fenster)
            alle_labels.append(label_name)

# Die Listen in finale Numpy-Arrays umwandeln
X_windows = np.array(alle_fenster)
y_windows = np.array(alle_labels)

print("\n--- WINDOWING FERTIG! ---")
print(f"Form der Feature-Matrix X: {X_windows.shape} -> ({X_windows.shape[0]} Fenster, jeweils {X_windows.shape[1]} Zeilen, {X_windows.shape[2]} Achsen)")
print(f"Form der Label-Liste y:    {y_windows.shape} -> ({y_windows.shape[0]} zugeordnete Klassen)")

Starte das Windowing über alle Dateien...

--- WINDOWING FERTIG! ---
Form der Feature-Matrix X: (68, 100, 3) -> (68 Fenster, jeweils 100 Zeilen, 3 Achsen)
Form der Label-Liste y:    (68,) -> (68 zugeordnete Klassen)


In [40]:
# Parameter für das Peak-Triggered Window laut Vorgabe
SCHWELLENWERT_G = 1.8  # Liegt perfekt zwischen 1,5 und 2,0 g
PUNKTE_VOR_PEAK = 50   # 1 Sekunde vor dem Aufprall
PUNKTE_NACH_PEAK = 100 # 2 Sekunden nach dem Aufprall
FENSTER_GROESSE_PEAK = PUNKTE_VOR_PEAK + PUNKTE_NACH_PEAK # 150 Punkte = 3 Sekunden

peak_fenster = []
peak_labels = []

print("Suche nach signifikanten Peaks für die Sturztyp-Klassifikation...")

for d in accelerometer_dateien:
    df_bereinigt = verarbeite_und_resample_datei(d)
    
    if df_bereinigt is not None:
        # 1. Gesamt-Beschleunigung (Betrag |a|) berechnen
        # Da manche Sensoren in m/s² und andere in g messen, normieren wir es hier grob,
        # falls die Werte extrem hoch sind (9.81 m/s² entspricht ca. 1g)
        x, y, z = df_bereinigt['x'].values, df_bereinigt['y'].values, df_bereinigt['z'].values
        betrag_a = np.sqrt(x**2 + y**2 + z**2)
        
        # Falls die App in m/s² misst, teilen wir durch 9.81, um auf "g" zu kommen
        if np.median(betrag_a) > 5.0:
            betrag_a = betrag_a / 9.80665
        
        # 2. Prüfen, ob der Schwellenwert überhaupt überschritten wurde
        if np.max(betrag_a) >= SCHWELLENWERT_G:
            # Den Index des absoluten Maximal-Peaks finden
            peak_idx = np.argmax(betrag_a)
            
            # Prüfen, ob wir genug Daten vor und nach dem Peak haben, um das Fenster zu schneiden
            if peak_idx - PUNKTE_VOR_PEAK >= 0 and peak_idx + PUNKTE_NACH_PEAK < len(df_bereinigt):
                start_idx = peak_idx - PUNKTE_VOR_PEAK
                end_idx = peak_idx + PUNKTE_NACH_PEAK
                
                # Sensorachsen für dieses 3-Sekunden-Fenster ausschneiden
                sensor_daten = df_bereinigt[['x', 'y', 'z']].values
                fenster = sensor_daten[start_idx:end_idx]
                
                peak_fenster.append(fenster)
                peak_labels.append(d.parent.name)

# In Numpy-Arrays umwandeln
X_peaks = np.array(peak_fenster)
y_peaks = np.array(peak_labels)

print("\n--- PEAK-WINDOWING FERTIG! ---")
print(f"Form der Peak-Feature-Matrix X: {X_peaks.shape} -> ({X_peaks.shape[0]} Peaks gefunden, jeweils {X_peaks.shape[1]} Zeilen, {X_peaks.shape[2]} Achsen)")
print(f"Form der Peak-Label-Liste y:    {y_peaks.shape}")

Suche nach signifikanten Peaks für die Sturztyp-Klassifikation...

--- PEAK-WINDOWING FERTIG! ---
Form der Peak-Feature-Matrix X: (3, 150, 3) -> (3 Peaks gefunden, jeweils 150 Zeilen, 3 Achsen)
Form der Peak-Label-Liste y:    (3,)


In [41]:
feature_liste = []

print("Starte Feature-Extraktion für die 68 Zeitfenster...")

# Wir gehen durch jedes der 68 Fenster (jedes Fenster hat die Form 100x3)
for i in range(len(X_windows)):
    fenster = X_windows[i]
    
    # Einzelne Achsen extrahieren
    x = fenster[:, 0]
    y = fenster[:, 1]
    z = fenster[:, 2]
    
    # 1. SIGNAL-VEKTOR-MAGNITUDE (SVM) berechnen
    svm = np.sqrt(x**2 + y**2 + z**2)
    
    # 2. RUCK (JERK) berechnen (erste Ableitung / Differenz zwischen Nachbarpunkten)
    jerk_x = np.diff(x)
    jerk_y = np.diff(y)
    jerk_z = np.diff(z)
    jerk_svm = np.diff(svm)
    
    # 3. STATISTISCHE MERKMALE BERECHNEN (für x, y, z, svm und jerks)
    features = {}
    
    # Funktion, um Statistiken für ein Signal kompakt zu sammeln
    def extrahiere_stats(signal_array, name):
        stats = {}
        stats[f'{name}_mean'] = np.mean(signal_array)
        stats[f'{name}_std'] = np.std(signal_array)
        stats[f'{name}_min'] = np.min(signal_array)
        stats[f'{name}_max'] = np.max(signal_array)
        stats[f'{name}_range'] = np.max(signal_array) - np.min(signal_array)
        stats[f'{name}_rms'] = np.sqrt(np.mean(signal_array**2))
        return stats

    # Statistiken für alle Signale berechnen und zusammenführen
    features.update(extrahiere_stats(x, 'x'))
    features.update(extrahiere_stats(y, 'y'))
    features.update(extrahiere_stats(z, 'z'))
    features.update(extrahiere_stats(svm, 'svm'))
    features.update(extrahiere_stats(jerk_svm, 'jerk_svm'))
    
    # Spezifisches sturzbezogenes Merkmal: Aufprallstärke (Peak)
    features['peak_impact'] = np.max(svm)
    
    feature_liste.append(features)

# Aus der Liste von Wörterbüchern eine wunderschöne strukturierte Tabelle machen
X_features_df = pd.DataFrame(feature_liste)

print("\n--- FEATURE-EXTRAKTION FERTIG! ---")
print(f"Form der fertigen Feature-Matrix (Tabelle): {X_features_df.shape}")
print(f"-> Wir haben aus den Kurven {X_features_df.shape[1]} mathematische Merkmale pro Fenster berechnet!")

Starte Feature-Extraktion für die 68 Zeitfenster...

--- FEATURE-EXTRAKTION FERTIG! ---
Form der fertigen Feature-Matrix (Tabelle): (68, 31)
-> Wir haben aus den Kurven 31 mathematische Merkmale pro Fenster berechnet!


In [42]:
# 1. Zielordner definieren und automatisch erstellen, falls er fehlt
ziel_ordner = Path("../data/processed")
ziel_ordner.mkdir(parents=True, exist_ok=True)

# 2. Die Labels (y) ebenfalls in eine saubere Tabelle umwandeln
y_features_df = pd.DataFrame(y_windows, columns=['label'])

# 3. Dateipfade für die Parquet-Dateien festlegen
pfad_X_parquet = ziel_ordner / "X_features.parquet"
pfad_y_parquet = ziel_ordner / "y_labels.parquet"

# 4. Als Parquet abspeichern
X_features_df.to_parquet(pfad_X_parquet, index=False)
y_features_df.to_parquet(pfad_y_parquet, index=False)

print("--- PIPELINE ERFOLGREICH ABGESCHLOSSEN! ---")
print(f"-> Feature-Matrix gespeichert unter: {pfad_X_parquet.resolve()}")
print(f"-> Labels gespeichert unter:         {pfad_y_parquet.resolve()}")
print("\nSo sehen die ersten 3 Zeilen deiner berechneten Feature-Tabelle aus:")
# Jetzt lassen wir uns die Tabelle auch einmal kurz anzeigen!
X_features_df.head(3)

--- PIPELINE ERFOLGREICH ABGESCHLOSSEN! ---
-> Feature-Matrix gespeichert unter: C:\Users\leahr\OneDrive\Machine Learning\Neuer Ordner\C-Team-ML4B-2026\data\processed\X_features.parquet
-> Labels gespeichert unter:         C:\Users\leahr\OneDrive\Machine Learning\Neuer Ordner\C-Team-ML4B-2026\data\processed\y_labels.parquet

So sehen die ersten 3 Zeilen deiner berechneten Feature-Tabelle aus:


,x_mean,x_std,x_min,x_max,x_range,x_rms,y_mean,y_std,y_min,y_max,...,svm_max,svm_range,svm_rms,jerk_svm_mean,jerk_svm_std,jerk_svm_min,jerk_svm_max,jerk_svm_range,jerk_svm_rms,peak_impact
0,-0.371563,2.194180,-13.440960,2.247528,15.688487,2.225418,0.837795,5.522680,-23.766922,10.192953,...,30.167849,29.755293,6.828410,0.202550,1.954066,-4.018406,15.243335,19.261741,1.964536,30.167849
1,0.053755,2.277998,-13.440960,2.903416,16.344376,2.278632,1.874875,6.413612,-23.766922,15.807321,...,30.167849,29.646458,8.473669,-0.026758,2.432998,-7.310848,15.243335,22.554183,2.433145,30.167849
2,-0.129353,1.685777,-7.081533,2.903416,9.984949,1.690732,1.175283,3.595153,-14.332884,15.807321,...,20.262370,19.868882,5.996465,-0.119837,1.955267,-7.310848,5.788149,13.098998,1.958936,20.262370


In [43]:
print("Folgende Dateien/Ordner haben die 3 Sturz-Peaks enthalten:")
print("-" * 60)

# Wir gehen durch die Liste der gefundenen Labels für die Peaks
for i, label in enumerate(peak_labels):
    print(f"Sturz {i+1}: {label}")

Folgende Dateien/Ordner haben die 3 Sturz-Peaks enthalten:
------------------------------------------------------------
Sturz 1: Brusttasche_Hinten_-2026-05-23_10-59-39 2
Sturz 2: Brusttasche_Seitw_rts_Rechts_-2026-05-23_10-57-42 2
Sturz 3: Brusttasche_Vorne_-2026-05-23_10-58-26 2
